In [ ]:
!pip install -q timm albumentations safetensors wandb huggingface_hub datasets scikit-learn

In [ ]:
import subprocess

result = subprocess.run(
    ["git", "clone", "https://github.com/hssling/anemia-detection-ai.git"],
    capture_output=True,
    text=True,
)
print(result.stdout)
import sys  # noqa: E402

sys.path.insert(0, "/kaggle/working/anemia-detection-ai")

In [ ]:
import os

from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _hf_token = _secrets.get_secret("HF_TOKEN")
    _wandb_key = _secrets.get_secret("WANDB_API_KEY")
    print("Loaded secrets via UserSecretsClient")
except Exception as exc:
    print(f"kaggle_secrets unavailable ({exc}), falling back to env vars")
    _hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    _wandb_key = os.environ.get("WANDB_API_KEY")

if not _hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty. In Kaggle: Add-ons -> Secrets -> Add Secret (name: HF_TOKEN) and enable it."
    )
if not _wandb_key:
    raise RuntimeError(
        "WANDB_API_KEY is empty. In Kaggle: Add-ons -> Secrets -> Add Secret (name: WANDB_API_KEY) and enable it."
    )

os.environ["HF_TOKEN"] = _hf_token.strip()
os.environ["WANDB_API_KEY"] = _wandb_key.strip()
login(token=os.environ["HF_TOKEN"])
import wandb  # noqa: E402

wandb.login(key=os.environ["WANDB_API_KEY"])
print("Auth complete")

In [ ]:
from datasets import load_dataset

VALID_CLASSES = {"normal", "mild", "moderate", "severe"}

def _usable(rows):
    out = []
    for row in rows:
        hb = row.get("hb_value")
        cls = row.get("anemia_class")
        if hb is None:
            continue
        try:
            hb = float(hb)
        except (TypeError, ValueError):
            continue
        if cls not in VALID_CLASSES:
            continue
        out.append(row)
    return out

import yaml

config = yaml.safe_load(open("/kaggle/working/anemia-detection-ai/training/config.yaml"))
dataset_repo = config["data"]["hf_dataset_repo"]
preferred_sources = set(config["data"].get("preferred_supervised_sources", []))
dataset = load_dataset(dataset_repo)
all_sites = set(r["site"] for r in dataset["train"]) | set(r["site"] for r in dataset["val"])
print("HF dataset repo:", dataset_repo)
print("Sites in dataset:", all_sites)
train_conj = [r for r in dataset["train"] if r["site"] == "conjunctiva"]
val_conj = [r for r in dataset["val"] if r["site"] == "conjunctiva"]
train_nail = [r for r in dataset["train"] if r["site"] == "nailbed"]
val_nail = [r for r in dataset["val"] if r["site"] == "nailbed"]
if not train_nail:
    for _alt in ("nail_bed", "nail-bed", "nail bed", "nailBed"):
        _t = [r for r in dataset["train"] if r["site"] == _alt]
        _v = [r for r in dataset["val"] if r["site"] == _alt]
        if _t:
            train_nail, val_nail = _t, _v
            print(f"Found nail-bed data under site='{_alt}'")
            break

if preferred_sources:
    _pref_train_conj = [r for r in train_conj if r.get("source_dataset") in preferred_sources]
    _pref_val_conj = [r for r in val_conj if r.get("source_dataset") in preferred_sources]
    if _pref_train_conj or _pref_val_conj:
        train_conj, val_conj = _pref_train_conj, _pref_val_conj
        print(f"Using preferred supervised conjunctiva sources: {sorted(preferred_sources)}")

usable_conj = _usable(train_conj + val_conj)
usable_nail = _usable(train_nail + val_nail)
print(f"Train conj: {len(train_conj)}, val conj: {len(val_conj)}, usable supervised conj: {len(usable_conj)}")
print(f"Train nail: {len(train_nail)}, val nail: {len(val_nail)}, usable supervised nail: {len(usable_nail)}")

if not usable_conj:
    raise RuntimeError(
        "No usable supervised conjunctiva rows found in the HF dataset. The current public registry appears to contain binary-only datasets without continuous Hb labels. Add Hb-labeled data (or ICMR field data) before running Hb-regression training."
    )

In [ ]:
import pathlib

from training.cross_validation import run_cross_validation  # noqa: E402

cv_summary_conj = run_cross_validation(
    rows=train_conj + val_conj,
    model_name="efficientnet_b4",
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/cv_conj"),
)
print("CV conjunctiva:", cv_summary_conj)

In [ ]:
from training.train import train_model

final_metrics_conj = train_model(
    model_name="efficientnet_b4",
    train_rows=train_conj + val_conj,
    val_rows=val_conj,
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/final"),
    fold=99,
    run_name="efficientnet_b4_conjunctiva_final",
)

In [ ]:
if not train_nail:
    print("No nail-bed training data found -- skipping nail-bed CV")
    cv_summary_nail = None
else:
    cv_summary_nail = run_cross_validation(
        rows=train_nail + val_nail,
        model_name="efficientnet_b4",
        config=config,
        output_dir=pathlib.Path("/kaggle/working/outputs/cv_nail"),
    )
    print("CV nail-bed:", cv_summary_nail)

In [ ]:
if not train_nail:
    print("No nail-bed training data found -- skipping nail-bed final training")
    final_metrics_nail = None
    nail_ckpt = None
else:
    final_metrics_nail = train_model(
        model_name="efficientnet_b4",
        train_rows=train_nail + val_nail,
        val_rows=val_nail,
        config=config,
        output_dir=pathlib.Path("/kaggle/working/outputs/final"),
        fold=98,
        run_name="efficientnet_b4_nailbed_final",
    )
    nail_ckpt = "/kaggle/working/outputs/final/efficientnet_b4_fold98_best.safetensors"
    print("Final nail-bed metrics:", final_metrics_nail)

In [ ]:
from training.models.ensemble import AnemiaEnsemble

conj_ckpt = "/kaggle/working/outputs/final/efficientnet_b4_fold99_best.safetensors"
if nail_ckpt is None:
    print("Nail-bed model absent -- using conjunctiva-only model (w_conj=1.0, w_nail=0.0)")
    w_conj, w_nail = 1.0, 0.0
else:
    w_conj, w_nail = AnemiaEnsemble.find_best_weights(
        conj_ckpt=conj_ckpt,
        nail_ckpt=nail_ckpt,
        val_rows_conj=val_conj,
        val_rows_nail=val_nail,
        config=config,
    )
    print(f"Best ensemble weights: w_conj={w_conj:.2f}, w_nail={w_nail:.2f}")

In [ ]:
from training.push_model_to_hf import push_all_models

push_all_models(
    conj_ckpt=conj_ckpt,
    nail_ckpt=nail_ckpt,
    cv_summary_conj=cv_summary_conj,
    cv_summary_nail=cv_summary_nail,
    w_conj=w_conj,
    w_nail=w_nail,
    config=config,
)
print("All models pushed to HuggingFace Hub")